In [1]:
from pathlib import Path
import sys

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

sys.path.insert(0, str(raiz))

from src.loaders.link_report import cargar_filas, tabla_calibracion

datos = raiz / "data" / "processed"
pares = tabla_calibracion(
    cargar_filas(
        historico_path=datos / "historico.sqlite",
        uc3m_path=datos / "uc3m.sqlite",
    )
)

print(f"Pares iniciales: {len(pares)}")

Pares iniciales: 39


In [2]:
transversales = {"14155", "14528", "17881"}

pares_filtrados = [
    par for par in pares
    if par.new_code not in transversales
]

numero_antes = len(pares_filtrados)

pares_unicos = {}
for par in pares_filtrados:
    clave = (par.new_code, par.dest_subject_id)
    if clave not in pares_unicos:
        pares_unicos[clave] = par

pares_filtrados = list(pares_unicos.values())
duplicados_eliminados = numero_antes - len(pares_filtrados)

print(f"Duplicados eliminados: {duplicados_eliminados}")

codigos_uc3m = {par.new_code for par in pares_filtrados}

print(f"Total de pares excluidos: {len(pares) - len(pares_filtrados)}")
print(f"Pares restantes: {len(pares_filtrados)}")
print(f"Asignaturas UC3M restantes: {len(codigos_uc3m)}")

Duplicados eliminados: 1
Total de pares excluidos: 4
Pares restantes: 35
Asignaturas UC3M restantes: 17


In [3]:
import random
from math import ceil

# Ordenamos antes de barajar para que el resultado sea reproducible.
codigos = sorted(codigos_uc3m)
random.Random(42).shuffle(codigos)

numero_holdout = ceil(0.20 * len(codigos))

codigos_holdout = set(codigos[:numero_holdout])
codigos_train = set(codigos[numero_holdout:])

pares_train = [
    par for par in pares_filtrados
    if par.new_code in codigos_train
]

pares_holdout = [
    par for par in pares_filtrados
    if par.new_code in codigos_holdout
]

assert codigos_train.isdisjoint(codigos_holdout)
assert len(pares_train) + len(pares_holdout) == len(pares_filtrados)

print(f"Train: {len(codigos_train)} asignaturas, {len(pares_train)} pares")
print(f"Hold-out: {len(codigos_holdout)} asignaturas, {len(pares_holdout)} pares")

Train: 13 asignaturas, 30 pares
Hold-out: 4 asignaturas, 5 pares


In [4]:
import numpy as np

generador = np.random.default_rng(42)

# Valores simulados entre 0.70 y 0.95.
cosenos_train = generador.uniform(0.70, 0.95, size=len(pares_train))

# Introducimos dos valores muy bajos para probar el filtro de outliers.
cosenos_train[0] = 0.20
cosenos_train[1] = 0.35

print("Cosenos inventados:")
print(np.round(cosenos_train, 3))

Cosenos inventados:
[0.2   0.35  0.915 0.874 0.724 0.944 0.89  0.897 0.732 0.813 0.793 0.932
 0.861 0.906 0.811 0.757 0.839 0.716 0.907 0.858 0.89  0.789 0.943 0.923
 0.895 0.749 0.817 0.711 0.739 0.871]


In [5]:
mediana = np.median(cosenos_train)
mad = np.median(np.abs(cosenos_train - mediana))
limite_outliers = mediana - 3 * mad

es_outlier = cosenos_train < limite_outliers
cosenos_limpios = cosenos_train[~es_outlier]

print(f"Mediana: {mediana:.4f}")
print(f"MAD: {mad:.4f}")
print(f"Límite inferior: {limite_outliers:.4f}")
print(f"Pares descartados: {es_outlier.sum()}")
print(f"Pares conservados: {len(cosenos_limpios)}")

print("\nPares descartados con cosenos SIMULADOS:")
for indice in np.flatnonzero(es_outlier):
    par = pares_train[indice]
    print(
        f"{par.new_code} → {par.catalogo_codigo}"
        f" | coseno: {cosenos_train[indice]:.3f}"
    )

Mediana: 0.8483
MAD: 0.0591
Límite inferior: 0.6709
Pares descartados: 2
Pares conservados: 28

Pares descartados con cosenos SIMULADOS:
14156 → BIOS 3753 | coseno: 0.200
14156 → BIOS 3755 | coseno: 0.350


In [6]:
percentil = 5
umbral = float(np.percentile(cosenos_limpios, percentil))

print(f"Percentil elegido: {percentil}")
print(f"Umbral con cosenos simulados: {umbral:.4f}")

print("\nResumen de calibración")
print(f"Pares iniciales: {len(pares)}")
print(
    f"Transversales excluidos: "
    f"{sum(par.new_code in transversales for par in pares)}"
)
print(f"Duplicados eliminados: {duplicados_eliminados}")
print(f"Train: {len(pares_train)} pares")
print(f"Hold-out: {len(pares_holdout)} pares")
print(f"Outliers de train: {es_outlier.sum()}")
print(f"Pares usados para calcular el umbral: {len(cosenos_limpios)}")

Percentil elegido: 5
Umbral con cosenos simulados: 0.7186

Resumen de calibración
Pares iniciales: 39
Transversales excluidos: 3
Duplicados eliminados: 1
Train: 30 pares
Hold-out: 5 pares
Outliers de train: 2
Pares usados para calcular el umbral: 28


In [7]:
from collections import Counter

recuento = Counter(
    (par.new_code, par.dest_subject_id)
    for par in pares_filtrados
)

duplicados = {
    clave: cantidad
    for clave, cantidad in recuento.items()
    if cantidad > 1
}

print(f"Pares únicos: {len(recuento)}")
print(f"Filas repetidas adicionales: {len(pares_filtrados) - len(recuento)}")
print(f"Duplicados: {duplicados}")
print(f"Duplicados eliminados: {duplicados_eliminados}")

Pares únicos: 35
Filas repetidas adicionales: 0
Duplicados: {}
Duplicados eliminados: 1


## Resultado y limitaciones

Se parte de 39 pares de Biomédica y Georgia Tech. Se excluyen 3 pares
con asignaturas transversales, siguiendo la issue #56. Se elimina una fila duplicada y quedan 35 pares distintos.

El reparto se realiza por código UC3M vigente, con semilla 42:
- Entrenamiento: 13 asignaturas y 30 pares.
- Hold-out: 4 asignaturas y 5 pares.

Se utilizan cosenos simulados para comprobar el procedimiento.
El filtro mediana − 3·MAD descarta los dos valores bajos introducidos
(0.20 y 0.35). Con los 28 pares restantes se calcula el percentil 5,
obteniendo un umbral de aproximadamente 0.7186.

El percentil 5 se elige como parámetro inicial dentro del intervalo
propuesto en la referencia técnica. El hold-out no interviene en
el cálculo del filtro ni del umbral.

Este umbral es de demostración. En la semana 6 se sustituirán los
cosenos simulados por los reales y se evaluará sobre el hold-out.
Sus 5 pares constituyen una muestra pequeña, por lo que las
métricas serán orientativas.